In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# 1. Synthesize Production-Grade Media Time-Series Data
np.random.seed(42)
dates = pd.date_range(start='2023-01-01', end='2025-12-31', freq='D')
n = len(dates)

# Base components: Secular Trend + Weekly/Annual Seasonality
trend = np.linspace(10000, 25000, n)
weekly_seasonality = 3000 * np.sin(2 * np.pi * dates.dayofweek / 7)
annual_seasonality = 5000 * np.sin(2 * np.pi * dates.dayofyear / 365.25)
noise = np.random.normal(0, 800, n)

# External Driver: Marketing Campaigns (Simulated Spends)
promo_spend = np.zeros(n)
promo_days = np.random.choice(n, size=40, replace=False)
for d in promo_days:
    promo_spend[d:d+4] = np.random.uniform(5000, 15000) # 4-day campaigns
promo_impact = promo_spend * 0.4

# Aggregate Target variable (Daily Media Views)
views = trend + weekly_seasonality + annual_seasonality + promo_impact + noise
df = pd.DataFrame({'Date': dates, 'Views': views, 'Promo_Spend': promo_spend})
df.set_index('Date', inplace=True)

# 2. Feature Engineering Pipeline
def create_features(data):
    df_feat = data.copy()
    # Calendar & Temporal features
    df_feat['dayofweek'] = df_feat.index.dayofweek
    df_feat['month'] = df_feat.index.month
    df_feat['dayofyear'] = df_feat.index.dayofyear

    # Historical Lag Features
    df_feat['lag_1'] = df_feat['Views'].shift(1)
    df_feat['lag_7'] = df_feat['Views'].shift(7)
    df_feat['lag_14'] = df_feat['Views'].shift(14)

    # Rolling Window Statistical Metrics
    df_feat['rolling_mean_7'] = df_feat['Views'].shift(1).rolling(window=7).mean()
    df_feat['rolling_std_7'] = df_feat['Views'].shift(1).rolling(window=7).std()
    return df_feat.dropna()

feature_df = create_features(df)

# 3. Temporal Train/Test Splitting (Holdout Strategy)
split_date = '2025-06-01'


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# 1. Synthesize Production-Grade Media Time-Series Data
np.random.seed(42)
dates = pd.date_range(start='2023-01-01', end='2025-12-31', freq='D')
n = len(dates)

# Base components: Secular Trend + Weekly/Annual Seasonality
trend = np.linspace(10000, 25000, n)
weekly_seasonality = 3000 * np.sin(2 * np.pi * dates.dayofweek / 7)
annual_seasonality = 5000 * np.sin(2 * np.pi * dates.dayofyear / 365.25)
noise = np.random.normal(0, 800, n)

# External Driver: Marketing Campaigns (Simulated Spends)
promo_spend = np.zeros(n)
promo_days = np.random.choice(n, size=40, replace=False)
for d in promo_days:
    promo_spend[d:d+4] = np.random.uniform(5000, 15000) # 4-day campaigns
promo_impact = promo_spend * 0.4

# Aggregate Target variable (Daily Media Views)
views = trend + weekly_seasonality + annual_seasonality + promo_impact + noise
df = pd.DataFrame({'Date': dates, 'Views': views, 'Promo_Spend': promo_spend})
df.set_index('Date', inplace=True)

# 2. Feature Engineering Pipeline
def create_features(data):
    df_feat = data.copy()
    # Calendar & Temporal features
    df_feat['dayofweek'] = df_feat.index.dayofweek
    df_feat['month'] = df_feat.index.month
    df_feat['dayofyear'] = df_feat.index.dayofyear

    # Historical Lag Features
    df_feat['lag_1'] = df_feat['Views'].shift(1)
    df_feat['lag_7'] = df_feat['Views'].shift(7)
    df_feat['lag_14'] = df_feat['Views'].shift(14)

    # Rolling Window Statistical Metrics
    df_feat['rolling_mean_7'] = df_feat['Views'].shift(1).rolling(window=7).mean()
    df_feat['rolling_std_7'] = df_feat['Views'].shift(1).rolling(window=7).std()
    return df_feat.dropna()

feature_df = create_features(df)

# 3. Temporal Train/Test Splitting (Holdout Strategy)
split_date = '2025-06-01'


In [2]:
train = feature_df.loc[feature_df.index < split_date]
test = feature_df.loc[feature_df.index >= split_date]

features = ['dayofweek', 'month', 'dayofyear', 'Promo_Spend',
            'lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7']
target = 'Views'

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# 4. Initialize and Train XGBoost Regressor Model
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

# 5. Generate Predictions and Calculate Evaluation Metrics
preds = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
wape = np.sum(np.abs(y_test - preds)) / np.sum(y_test) * 100

print(f'Execution Performance Verification:')
print(f'Test Root Mean Squared Error (RMSE): {rmse:.2f} Views')
print(f'Test Weighted Absolute Percentage Error (WAPE): {wape:.2f}%')


Execution Performance Verification:
Test Root Mean Squared Error (RMSE): 1175.32 Views
Test Weighted Absolute Percentage Error (WAPE): 4.26%
